# Day 5 · Notebook 2 of 5 – Final regression

**What this notebook does:** re-checks the whole pipeline one last time before UAT and saves every result to `capstone_5iot.gold.regression_results`.

| Section | What is checked |
|---|---|
| 1 | Row counts in Bronze, Silver and Gold |
| 2 | Reconciliation: Bronze = Silver + rejected, Gold = Silver |
| 3 | Business keys are unique |
| 4 | No orphan keys (every foreign key exists in its parent table) |
| 5 | DQ rules and quarantine traceability |
| 6 | Chronology (timestamps in the right order, valid status changes) |
| 7 | KPI reconciliation (mart = facts = independent SQL) |
| 8 | Dashboard views in Gold give the same totals as the Gold tables |
| 9 | Save results and compare with the Day 4 test run |

**Run after** `7_1_dashboard_views`. Bronze is only read here, never written.

**Note on expected values:** the row counts (11,653, 11,566, 665 …) are fixed to the delivered data set. The other checks are rule-based (for example "Bronze = Silver + rejected", "0 orphans") and stay valid when new data arrives.

## Setup – names, a result list and a small `check` helper

In [0]:
from pyspark.sql import functions as F

CATALOG = "capstone_5iot"
BRONZE  = f"{CATALOG}.iot_schema"
SILVER  = f"{CATALOG}.silver"
GOLD    = f"{CATALOG}.gold"

ABNORMAL = "(temperature_c > 85 OR vibration_mm_s > 7.1 OR pressure_bar > 18)"   # abnormal reading rule

RUN_TS  = spark.sql("SELECT date_trunc('SECOND', current_timestamp())").first()[0]
results = []          # every check adds one row here


def q(sql):
    """Run a SQL query and return the first value of the first row."""
    return spark.sql(sql).first()[0]


def check(test_id, layer, category, description, actual, expected):
    """Compare actual with expected and store the result. Numbers with decimals are compared with a small tolerance."""
    if isinstance(actual, float) or isinstance(expected, float):
        ok = actual is not None and abs(float(actual) - float(expected)) < 1e-5
    else:
        ok = actual == expected
    status = "PASS" if ok else "FAIL"
    results.append((test_id, layer, category, description, str(expected), str(actual), status))
    if status == "FAIL":
        print(f"FAIL {test_id} {description}: expected={expected} actual={actual}")


def layer_of(table):
    return {"iot_schema": "bronze", "silver": "silver", "gold": "gold"}[table.split(".")[1]]


def summary(prefix):
    part = [r for r in results if r[0].startswith(prefix)]
    passed = sum(r[6] == "PASS" for r in part)
    print(f"{prefix}: {passed} of {len(part)} checks PASS")

print("Regression run:", RUN_TS)

## 1. Row counts

In [0]:
expected_rows = {
    f"{BRONZE}.bronze_sensor_readings": 11653,
    f"{BRONZE}.bronze_failure_events": 665,
    f"{BRONZE}.bronze_work_orders": 335,
    f"{BRONZE}.bronze_machine_master": 150,
    f"{BRONZE}.bronze_plant_master": 159,
    f"{BRONZE}.bronze_failure_code_lookup": 168,
    f"{BRONZE}.bronze_work_order_status_cdc": 652,
    f"{SILVER}.silver_sensor_readings": 11566,
    f"{SILVER}.silver_rejected_records": 87,
    f"{SILVER}.silver_failure_events": 665,
    f"{SILVER}.silver_work_orders": 335,
    f"{SILVER}.silver_work_order_status_history": 652,
    f"{SILVER}.silver_work_order_current": 335,
    f"{SILVER}.silver_maintenance_episodes": 665,
    f"{SILVER}.silver_machine_master": 150,
    f"{SILVER}.silver_plant_master": 159,
    f"{SILVER}.silver_failure_code_lookup": 168,
    f"{SILVER}.dq_rule_catalogue": 12,
    f"{GOLD}.dim_machine": 150,
    f"{GOLD}.dim_plant": 159,
    f"{GOLD}.dim_failure_code": 168,
    f"{GOLD}.fact_sensor_activity": 11566,
    f"{GOLD}.fact_maintenance_episodes": 665,
    f"{GOLD}.fact_status_transitions": 652,
}
for i, (table, expected) in enumerate(expected_rows.items(), 1):
    check(f"R1-{i:02d}", layer_of(table), "row_count", f"{table} rows", spark.table(table).count(), expected)

# these tables have no fixed size, they only must not be empty
must_have_rows = [f"{SILVER}.silver_load_audit", f"{SILVER}.silver_sensor_windows", f"{SILVER}.silver_telemetry_gaps",
                  f"{GOLD}.dim_date", f"{GOLD}.mart_sensor_daily"]
for i, table in enumerate(must_have_rows, 1):
    check(f"R1-N{i:02d}", layer_of(table), "row_count", f"{table} is not empty", spark.table(table).count() > 0, True)

summary("R1")

## 2. Reconciliation
Every Bronze row must end up either in Silver or in the quarantine table. Gold facts and dimensions must have the same rows as their Silver source.

In [0]:
REJ = f"{SILVER}.silver_rejected_records"

bronze_to_silver = [("bronze_sensor_readings", "silver_sensor_readings"),
                    ("bronze_failure_events",  "silver_failure_events"),
                    ("bronze_work_orders",     "silver_work_orders")]
for i, (b, s) in enumerate(bronze_to_silver, 1):
    bronze_rows = spark.table(f"{BRONZE}.{b}").count()
    silver_rows = spark.table(f"{SILVER}.{s}").count()
    rejected    = q(f"SELECT count(*) FROM {REJ} WHERE source_table = '{b}'")
    check(f"R2-{i:02d}", "silver", "reconciliation", f"{b} = {s} + rejected", silver_rows + rejected, bronze_rows)

check("R2-04", "silver", "reconciliation", "silver_load_audit rows not reconciled",
      q(f"SELECT count_if(NOT reconciled) FROM {SILVER}.silver_load_audit"), 0)
check("R2-05", "silver", "reconciliation", "status history rows = Bronze CDC rows",
      spark.table(f"{SILVER}.silver_work_order_status_history").count(),
      spark.table(f"{BRONZE}.bronze_work_order_status_cdc").count())

silver_to_gold = [("fact_sensor_activity",      "silver_sensor_readings"),
                  ("fact_maintenance_episodes", "silver_maintenance_episodes"),
                  ("fact_status_transitions",   "silver_work_order_status_history"),
                  ("dim_machine",               "silver_machine_master"),
                  ("dim_plant",                 "silver_plant_master"),
                  ("dim_failure_code",          "silver_failure_code_lookup")]
for i, (g, s) in enumerate(silver_to_gold, 6):
    check(f"R2-{i:02d}", "gold", "reconciliation", f"gold.{g} rows = silver.{s} rows",
          spark.table(f"{GOLD}.{g}").count(), spark.table(f"{SILVER}.{s}").count())

summary("R2")

## 3. Unique business keys
Duplicate keys = rows − distinct keys. It must be 0 for every table.

In [0]:
keys = [
    (f"{SILVER}.silver_sensor_readings",           "reading_id"),
    (f"{SILVER}.silver_failure_events",            "failure_event_id"),
    (f"{SILVER}.silver_work_orders",               "work_order_id"),
    (f"{SILVER}.silver_work_order_current",        "work_order_id"),
    (f"{SILVER}.silver_work_order_status_history", "work_order_id, sequence_no"),
    (f"{SILVER}.silver_maintenance_episodes",      "failure_event_id"),
    (f"{SILVER}.silver_machine_master",            "machine_id"),
    (f"{SILVER}.silver_plant_master",              "plant_id"),
    (f"{SILVER}.silver_failure_code_lookup",       "failure_code"),
    (f"{SILVER}.dq_rule_catalogue",                "rule_id"),
    (f"{GOLD}.dim_machine",                        "machine_sk"),
    (f"{GOLD}.dim_plant",                          "plant_sk"),
    (f"{GOLD}.dim_failure_code",                   "failure_code_sk"),
    (f"{GOLD}.dim_date",                           "date_key"),
    (f"{GOLD}.fact_sensor_activity",               "reading_id"),
    (f"{GOLD}.fact_maintenance_episodes",          "failure_event_id"),
    (f"{GOLD}.fact_status_transitions",            "work_order_id, sequence_no"),
    (f"{GOLD}.mart_sensor_daily",                  "machine_id, activity_date"),
]
for i, (table, key) in enumerate(keys, 1):
    duplicates = q(f"SELECT count(*) - count(DISTINCT struct({key})) FROM {table}")
    check(f"R3-{i:02d}", layer_of(table), "uniqueness", f"{table} duplicate keys on ({key})", duplicates, 0)

summary("R3")

## 4. No orphan keys
Rows in the child table whose key is missing in the parent table. It must be 0.

In [0]:
FSA, FME, FST, MART = (f"{GOLD}.fact_sensor_activity", f"{GOLD}.fact_maintenance_episodes",
                       f"{GOLD}.fact_status_transitions", f"{GOLD}.mart_sensor_daily")
DM, DP, DF, DD = f"{GOLD}.dim_machine", f"{GOLD}.dim_plant", f"{GOLD}.dim_failure_code", f"{GOLD}.dim_date"

# (child table, child column, parent table, parent column)
foreign_keys = [
    (FSA, "machine_sk", DM, "machine_sk"), (FSA, "plant_sk", DP, "plant_sk"), (FSA, "date_key", DD, "date_key"),
    (FME, "machine_sk", DM, "machine_sk"), (FME, "plant_sk", DP, "plant_sk"),
    (FME, "failure_code_sk", DF, "failure_code_sk"), (FME, "date_key", DD, "date_key"),
    (FST, "machine_sk", DM, "machine_sk"), (FST, "date_key", DD, "date_key"),
    (MART, "machine_sk", DM, "machine_sk"), (MART, "plant_sk", DP, "plant_sk"), (MART, "date_key", DD, "date_key"),
    (DM, "plant_sk", DP, "plant_sk"),
    (f"{SILVER}.silver_sensor_readings", "machine_id",    f"{SILVER}.silver_machine_master",      "machine_id"),
    (f"{SILVER}.silver_failure_events",  "machine_id",    f"{SILVER}.silver_machine_master",      "machine_id"),
    (f"{SILVER}.silver_failure_events",  "work_order_id", f"{SILVER}.silver_work_orders",         "work_order_id"),
    (f"{SILVER}.silver_failure_events",  "failure_code",  f"{SILVER}.silver_failure_code_lookup", "failure_code"),
    (f"{SILVER}.silver_work_orders",     "machine_id",    f"{SILVER}.silver_machine_master",      "machine_id"),
]
for i, (child, fk, parent, pk) in enumerate(foreign_keys, 1):
    orphans = q(f"SELECT count(*) FROM {child} c LEFT ANTI JOIN {parent} p ON c.{fk} = p.{pk}")
    check(f"R4-{i:02d}", layer_of(child), "referential_integrity", f"{child}.{fk} orphans vs {parent}", orphans, 0)

summary("R4")

## 5. DQ rules and quarantine traceability
* The rule catalogue has 12 rules, and every rejected row points to one of them.
* Rejected rows keep their source file and raw record, so each one can be traced back to Bronze.
* Rejects and flags match the Day 2 results (41 duplicates, 46 invalid temperatures, …).

In [0]:
SS, SWO = f"{SILVER}.silver_sensor_readings", f"{SILVER}.silver_work_orders"

check("R5-01", "silver", "dq_rule", "rules in dq_rule_catalogue", spark.table(f"{SILVER}.dq_rule_catalogue").count(), 12)

rejects_by_rule = {r.rule_id: r.n for r in spark.sql(f"SELECT rule_id, count(*) n FROM {REJ} GROUP BY rule_id").collect()}
check("R5-02", "silver", "dq_rule", "rejected rows by rule (DQ04 temp, DQ05 duplicate)", rejects_by_rule, {"DQ04": 46, "DQ05": 41})

check("R5-03", "silver", "quarantine_traceability", "rejected rows whose rule is not in the catalogue",
      q(f"SELECT count(*) FROM {REJ} r LEFT ANTI JOIN {SILVER}.dq_rule_catalogue c ON r.rule_id = c.rule_id"), 0)
check("R5-04", "silver", "quarantine_traceability", "rejected rows without source file",
      q(f"SELECT count_if(_source_file IS NULL OR _source_file = '') FROM {REJ}"), 0)
check("R5-05", "silver", "quarantine_traceability", "rejected rows without raw record, source table or reason",
      q(f"SELECT count_if(_raw_record IS NULL OR source_table IS NULL OR reason IS NULL) FROM {REJ}"), 0)
check("R5-06", "silver", "quarantine_traceability", "rejected sensor rows not found in Bronze (same id and file)", q(f"""
      SELECT count(*) FROM {REJ} r
      WHERE r.source_table = 'bronze_sensor_readings'
        AND NOT EXISTS (SELECT 1 FROM {BRONZE}.bronze_sensor_readings b
                        WHERE b.reading_id = r.record_key AND b._source_file = r._source_file)"""), 0)

# flags: rows are kept in Silver but marked
check("R5-07", "silver", "dq_rule", "DQ06 rows with missing optional fields", q(f"SELECT count_if(has_missing_optional) FROM {SS}"), 14)
check("R5-08", "silver", "dq_rule", "DQ07 readings before commissioning", q(f"SELECT count_if(is_before_commissioning) FROM {SS}"), 2045)
check("R5-09", "silver", "dq_rule", "DQ08 readings from INACTIVE machines", q(f"SELECT count_if(is_inactive_machine) FROM {SS}"), 3382)
check("R5-10", "silver", "dq_rule", "DQ11 completed_ts on a non-COMPLETED order", q(f"SELECT count_if(is_completed_ts_invalid) FROM {SWO}"), 0)

# DQ12: the current status must always be the latest sequence number (old changes never overwrite new ones)
check("R5-11", "silver", "dq_rule", "DQ12 current status is not the latest change", q(f"""
      SELECT count(*) FROM {SILVER}.silver_work_order_current c
      JOIN (SELECT work_order_id, max(sequence_no) AS latest
            FROM {SILVER}.silver_work_order_status_history GROUP BY work_order_id) h USING (work_order_id)
      WHERE c.sequence_no <> h.latest"""), 0)

summary("R5")

## 6. Chronology
Timestamps must be in a logical order and work orders may only move OPEN → IN_PROGRESS → COMPLETED.

In [0]:
SFE, HIST = f"{SILVER}.silver_failure_events", f"{SILVER}.silver_work_order_status_history"

valid_transition = """(coalesce(from_status, 'NULL') = 'NULL'        AND to_status = 'OPEN')
                     OR (from_status = 'OPEN'        AND to_status = 'IN_PROGRESS')
                     OR (from_status = 'IN_PROGRESS' AND to_status = 'COMPLETED')"""

chronology = [
    ("R6-01", "silver", "work orders with created_ts > planned_start_ts", f"SELECT count_if(created_ts > planned_start_ts) FROM {SWO}"),
    ("R6-02", "silver", "work orders with planned_start_ts > completed_ts", f"SELECT count_if(planned_start_ts > completed_ts_clean) FROM {SWO}"),
    ("R6-03", "silver", "work orders with created_ts > completed_ts", f"SELECT count_if(created_ts > completed_ts_clean) FROM {SWO}"),
    ("R6-04", "silver", "COMPLETED work orders without completed_ts", f"SELECT count_if(work_order_status = 'COMPLETED' AND completed_ts_clean IS NULL) FROM {SWO}"),
    ("R6-05", "silver", "failure events with NULL failure_ts", f"SELECT count_if(failure_ts IS NULL) FROM {SFE}"),
    ("R6-06", "silver", "failure events before 2000 or in the future", f"SELECT count_if(failure_ts < '2000-01-01' OR failure_ts > current_timestamp()) FROM {SFE}"),
    ("R6-07", "gold",   "invalid status transitions", f"SELECT count_if(NOT ({valid_transition})) FROM {FST}"),
    ("R6-08", "silver", "status history: later sequence_no with an earlier timestamp", f"""
        SELECT count_if(bad) FROM (
          SELECT effective_ts < lag(effective_ts) OVER (PARTITION BY work_order_id ORDER BY sequence_no) AS bad FROM {HIST})"""),
]
for test_id, layer, description, sql in chronology:
    check(test_id, layer, "chronology", description, q(sql), 0)

summary("R6")

## 7. KPI reconciliation
The daily mart must give the same totals as the facts, and the KPIs must match an independent calculation.

In [0]:
mart = spark.sql(f"""SELECT sum(reading_count) AS readings, sum(abnormal_count) AS abnormal,
                             sum(failure_count) AS failures, sum(downtime_minutes) AS downtime
                      FROM {MART}""").first()

check("R7-01", "gold", "kpi", "mart readings = fact_sensor_activity rows", mart.readings, spark.table(FSA).count())
check("R7-02", "gold", "kpi", "mart abnormal = abnormal recomputed from thresholds", mart.abnormal, q(f"SELECT count_if({ABNORMAL}) FROM {FSA}"))
check("R7-03", "gold", "kpi", "mart failures = fact_maintenance_episodes rows", mart.failures, spark.table(FME).count())
check("R7-04", "gold", "kpi", "mart downtime = fact downtime", mart.downtime, q(f"SELECT sum(downtime_minutes) FROM {FME}"))
check("R7-05", "gold", "kpi", "total downtime = client figure 3,349 min", mart.downtime, 3349)

check("R7-06", "gold", "kpi", "machine-days where availability <> (1440 - downtime) / 1440", q(f"""
      WITH dt AS (SELECT machine_id, to_date(failure_ts) AS d, sum(downtime_minutes) AS dm FROM {FME} GROUP BY ALL)
      SELECT count(*) FROM {MART} m LEFT JOIN dt ON m.machine_id = dt.machine_id AND m.activity_date = dt.d
      WHERE m.availability IS NULL OR abs(m.availability - round((1440 - coalesce(dt.dm, 0)) / 1440, 4)) > 1e-9"""), 0)
check("R7-07", "gold", "kpi", "availability outside 0..1", q(f"SELECT count_if(availability < 0 OR availability > 1) FROM {MART}"), 0)
check("R7-08", "gold", "kpi", "overall abnormal rate: mart vs Silver",
      round(mart.abnormal / mart.readings, 4), round(q(f"SELECT count_if({ABNORMAL}) / count(*) FROM {SS}"), 4))

summary("R7")

## 8. Dashboard views reconcile to Gold
Every view that carries a total must give back the Gold figures: downtime **3,349 min**, failures **665**, valid readings **11,566**.

In [0]:
def view(name):
    return f"{GOLD}.{name}"

DOWNTIME, FAILURES, READINGS, MACHINES = 3349, 665, 11566, 150
DAYS = spark.table(DD).count()
head = spark.table(view("vw_kpi_headline")).first()

dashboard_checks = [
    ("R8-01", "vw_kpi_headline downtime",            head.total_downtime_minutes, DOWNTIME),
    ("R8-02", "vw_kpi_headline failures",            head.failure_count, FAILURES),
    ("R8-03", "vw_kpi_headline valid readings",      head.valid_readings, READINGS),
    ("R8-04", "vw_kpi_headline machines",            head.machines_monitored, MACHINES),
    ("R8-05", "vw_kpi_headline availability = 1 - downtime / (machines x days x 1440)",
              head.overall_availability, round(1 - DOWNTIME / (MACHINES * DAYS * 1440), 6)),
    ("R8-06", "vw_daily_trend downtime",             q(f"SELECT sum(downtime_minutes) FROM {view('vw_daily_trend')}"), DOWNTIME),
    ("R8-07", "vw_daily_trend failures",             q(f"SELECT sum(failures) FROM {view('vw_daily_trend')}"), FAILURES),
    ("R8-08", "vw_daily_trend readings",             q(f"SELECT sum(valid_readings) FROM {view('vw_daily_trend')}"), READINGS),
    ("R8-09", "vw_daily_trend rows = days",          spark.table(view("vw_daily_trend")).count(), DAYS),
    ("R8-10", "vw_machine_reliability downtime",     q(f"SELECT sum(downtime_minutes) FROM {view('vw_machine_reliability')}"), DOWNTIME),
    ("R8-11", "vw_machine_reliability rows = machines", spark.table(view("vw_machine_reliability")).count(), MACHINES),
    ("R8-12", "vw_plant_summary downtime",           q(f"SELECT sum(downtime_minutes) FROM {view('vw_plant_summary')}"), DOWNTIME),
    ("R8-13", "vw_plant_summary machines",           q(f"SELECT sum(machines) FROM {view('vw_plant_summary')}"), MACHINES),
    ("R8-14", "vw_failure_pareto downtime",          q(f"SELECT sum(downtime_minutes) FROM {view('vw_failure_pareto')}"), DOWNTIME),
    ("R8-15", "vw_failure_pareto failures",          q(f"SELECT sum(failures) FROM {view('vw_failure_pareto')}"), FAILURES),
    ("R8-16", "vw_sensor_drift readings",            q(f"SELECT sum(readings) FROM {view('vw_sensor_drift')}"), READINGS),
    ("R8-17", "vw_failure_correlation readings",     q(f"SELECT sum(readings) FROM {view('vw_failure_correlation')}"), READINGS),
    ("R8-18", "vw_maintenance work orders",          q(f"SELECT sum(work_orders) FROM {view('vw_maintenance')}"), spark.table(SWO).count()),
    ("R8-19", "vw_maintenance linked downtime",      q(f"SELECT sum(linked_downtime_minutes) FROM {view('vw_maintenance')}"), DOWNTIME),
    ("R8-20", "vw_data_quality rejected records",
              q(f"SELECT sum(metric_value) FROM {view('vw_data_quality')} WHERE metric_group = 'rejected_by_rule'"), 87.0),
]
for test_id, description, actual, expected in dashboard_checks:
    check(test_id, "gold", "dashboard_reconciliation", description, actual, expected)

summary("R8")

## 9. Save the results and compare with Day 4
Results are added to `gold.regression_results` with this run's timestamp. Re-running this cell replaces the rows of the same run (no duplicates).

In [0]:
REG = f"{GOLD}.regression_results"
spark.sql(f"""CREATE TABLE IF NOT EXISTS {REG} (
    test_id STRING, layer STRING, category STRING, description STRING,
    expected STRING, actual STRING, status STRING, run_ts TIMESTAMP)
  COMMENT 'Day 5 final regression - one row per check per run'""")

run_df = (spark.createDataFrame(results, "test_id string, layer string, category string, description string, "
                                         "expected string, actual string, status string")
               .withColumn("run_ts", F.lit(RUN_TS)))

spark.sql(f"DELETE FROM {REG} WHERE run_ts = timestamp'{RUN_TS}'")     # makes the cell safe to re-run
run_df.write.mode("append").saveAsTable(REG)

passed = run_df.filter("status = 'PASS'").count()
print(f"Final regression: {passed} of {run_df.count()} checks PASS")
display(run_df.groupBy("category").agg(F.count("*").alias("checks"),
                                        F.count_if(F.col("status") == "PASS").alias("passed")).orderBy("category"))
display(run_df.filter("status <> 'PASS'"))

In [0]:
# Day 4 baseline = the latest run in gold.test_results
day4 = spark.sql(f"""SELECT 'Day 4 test matrix' AS run, max(run_ts) AS run_ts, count(*) AS checks,
                             count_if(status = 'PASS') AS passed, count_if(status = 'FAIL') AS failed
                      FROM {GOLD}.test_results
                      WHERE run_ts = (SELECT max(run_ts) FROM {GOLD}.test_results)""")
day5 = spark.sql(f"""SELECT 'Day 5 final regression' AS run, max(run_ts) AS run_ts, count(*) AS checks,
                             count_if(status = 'PASS') AS passed, count_if(status = 'FAIL') AS failed
                      FROM {REG}
                      WHERE run_ts = timestamp'{RUN_TS}'""")
display(day4.unionByName(day5).withColumn("pass_rate", F.round(F.col("passed") / F.col("checks"), 4)))